# 🔬 SSL Mixte → Phase 2/3 sur .npy SEUL
## Suggestion Dr. Sarun (piste 1) — l'exposition SSL au CSV suffit-elle ?
### CMKL University · Stage 2026

---

**Hypothèse testée** : le backbone a vu du CSV pendant la Phase 1 (SSL, sans
aucun label). Si on entraîne ensuite Phase 2 et 3 **uniquement sur .npy**
(jamais de classification sur CSV), est-ce que la simple exposition non
supervisée en Phase 1 suffit à éviter l'effondrement du domain gap qu'on
observait avec le modèle .npy pur (23.47% sur CSV) ?

**Ce notebook réutilise `ssl_backbone_mixed.pth` déjà entraîné** — pas besoin
de refaire la Phase 1, on enchaîne directement sur Phase 2/3.

**Comparaison à trois termes** :
```
                        SSL          Phase 2/3      Test npy   Test CSV
Modèle .npy seul      : npy only     npy only       95.25%     23.47%
Modèle Mixte          : npy+CSV      npy+CSV        94.62%     99.25%
CE NOTEBOOK           : npy+CSV      npy ONLY       ?          ?
```

Si ce nouveau modèle est proche du **Mixte** sur CSV (au lieu de proche du
**.npy seul**), ça validerait que la simple exposition SSL (même sans
jamais classifier de CSV) suffit à "immuniser" contre le domain gap —
un résultat très intéressant pour Dr. Sarun.


---
## ⚙️ Section 0 — Imports & Configuration


In [1]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
for pkg in ['scikit-learn', 'seaborn']:
    try: __import__(pkg.replace('-','_'))
    except ImportError: install(pkg)
print('✓ Packages prêts')

✓ Packages prêts


In [2]:
import os, glob, re, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HOME   = os.path.expanduser('~')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cuda
GPU    : NVIDIA A100-SXM4-40GB


In [3]:
from torch.utils.tensorboard import SummaryWriter

LOG_DIR = os.path.join(HOME, 'runs', 'patchtst_sslmixed_npyonly')
os.makedirs(LOG_DIR, exist_ok=True)
writer  = SummaryWriter(LOG_DIR)
print(f'✓ TensorBoard logs → {LOG_DIR}')

def safe_log(writer, *args, method='add_scalar', **kwargs):
    try: getattr(writer, method)(*args, **kwargs)
    except Exception: pass

✓ TensorBoard logs → /home/glider/runs/patchtst_sslmixed_npyonly


In [4]:
CFG = {
    'L'         : 6700,
    'WN_MIN'    : 650,
    'WN_MAX'    : 4000,
    'WN_STEP'   : 0.5,

    'patch_size' : 320,
    'stride'     : 256,

    'd_model'   : 256,
    'n_heads'   : 16,
    'n_layers'  : 5,
    'd_ff'      : 512,
    'dropout'   : 0.1,

    'alpha'        : 1.0,
    'beta'         : 0.5,
    'probe_epochs' : 80,
    'probe_lr'     : 1e-3,
    'ft_epochs'    : 80,
    'ft_lr'        : 1e-5,
    'batch_size'   : 32,

    'ssl_path'   : os.path.join(HOME, 'models', 'ssl_backbone_mixed.pth'),   # RÉUTILISÉ, pas recréé
    'probe_path' : os.path.join(HOME, 'models', 'probe_model_sslmixed_npyonly.pth'),
    'final_path' : os.path.join(HOME, 'models', 'final_model_sslmixed_npyonly.pth'),

    'noise_variant' : 'Upto30SNR',
}
os.makedirs(os.path.join(HOME, 'models'), exist_ok=True)

WN_GRID   = np.arange(CFG['WN_MIN'], CFG['WN_MAX'], CFG['WN_STEP'])
CFG['L']  = len(WN_GRID)
L = CFG['L']
N_PATCHES = (L - CFG['patch_size']) // CFG['stride'] + 2
print(f'L = {L}, N_PATCHES = {N_PATCHES}')
assert CFG['d_model'] % CFG['n_heads'] == 0

L = 6700, N_PATCHES = 26


In [5]:
ASSUMED_CLASSES = [
    'ABS', 'ACRYLIC', 'CELLULOSE', 'CHITOSAN', 'ENR', 'EPDM', 'EVA', 'HDPE',
    'LDPE', 'NYLON', 'PBAT', 'PBS', 'PC', 'PEEK', 'PEI', 'PET',
    'PF THERMOPLASTIC', 'PF THERMOSET', 'PHB', 'PLA', 'PMMA', 'POM', 'PP',
    'PS', 'PTFE', 'PU', 'PVA', 'PVC', 'PVDF', 'SAN',
]
N_CLASSES = len(ASSUMED_CLASSES)
CFG['N_CLASSES'] = N_CLASSES
le = LabelEncoder()
le.fit(ASSUMED_CLASSES)
print(f'{N_CLASSES} classes')

30 classes


---
## 📊 Section 1 — Données .npy (Train + Test — c'est la SEULE source pour Phase 2/3)


In [6]:
NPY_ROOT  = os.path.join(HOME, 'data', '2026-FTIR-Preprocesed','2026 - FTIR - 4. Selected Datasets - Preprocessed')
TRAIN_DIR = os.path.join(NPY_ROOT, '1.1 TrainingSet - UptoY dB')
TEST_DIR  = os.path.join(NPY_ROOT, '1.2 TestSet - UptoY dB')

def npy_path(base_dir, filename):
    p = os.path.join(base_dir, filename)
    if not os.path.exists(p): print(f'  ✗ INTROUVABLE : {p}')
    return p

noise = CFG['noise_variant']
npy_train_clean = np.load(npy_path(TRAIN_DIR, 'TrainGroundTruthSet_Pre.npy'))
npy_train_noisy = np.load(npy_path(TRAIN_DIR, f'TrainNoisySet_{noise}_Pre.npy'))
npy_test_clean  = np.load(npy_path(TEST_DIR,  'TestGroundTruthSet_Pre.npy'))
npy_test_noisy  = np.load(npy_path(TEST_DIR,  f'TestNoisySet_{noise}_Pre.npy'))

assert npy_train_clean.shape[1] == L
N_PER_CLASS_TRAIN = npy_train_clean.shape[0] // N_CLASSES
N_PER_CLASS_TEST  = npy_test_clean.shape[0]  // N_CLASSES
labels_train_full = np.repeat(np.arange(N_CLASSES), N_PER_CLASS_TRAIN)
labels_test        = np.repeat(np.arange(N_CLASSES), N_PER_CLASS_TEST)

print(f'.npy Train : {npy_train_noisy.shape[0]}')
print(f'.npy Test  : {npy_test_noisy.shape[0]}')

.npy Train : 18000
.npy Test  : 18000


In [7]:
# ── Split Train/Val .npy (identique à tous les autres notebooks) ──────────
N_VAL_PER_CLASS = 30
val_idx, train_idx = [], []
for c in range(N_CLASSES):
    cls_idx = np.where(labels_train_full == c)[0]
    rng = np.random.RandomState(SEED)
    rng.shuffle(cls_idx)
    val_idx.extend(cls_idx[:N_VAL_PER_CLASS])
    train_idx.extend(cls_idx[N_VAL_PER_CLASS:])
val_idx   = np.array(val_idx)
train_idx = np.array(train_idx)

X_train_noisy = npy_train_noisy[train_idx]
X_train_clean = npy_train_clean[train_idx]
y_train       = labels_train_full[train_idx]

X_val_noisy = npy_train_noisy[val_idx]
X_val_clean = npy_train_clean[val_idx]
y_val       = labels_train_full[val_idx]

print(f'Train : {len(train_idx)}   Val : {len(val_idx)}   Test : {len(npy_test_noisy)} (officiel)')

Train : 17100   Val : 900   Test : 18000 (officiel)


---
## 📁 Section 2 — Données CSV (ÉVALUATION UNIQUEMENT — jamais utilisées pour entraîner Phase 2/3)

⚠️ Point clé de ce test : le CSV n'apparaît QUE dans le test set fixe, jamais
dans le train de Phase 2/3. On veut savoir si le modèle généralise dessus
malgré ça, uniquement grâce à son exposition non supervisée en Phase 1.


In [8]:
CSV_ROOT = os.path.join(HOME, 'data', '2026-FirstDataSet', '2026 - Complete FTIR Dataset')
PATHS_CSV = {
    '2023_base' : os.path.join(CSV_ROOT, '2023 Dataset - 22 MP Types with 10 Clean and 60 Noisy'),
    '2025_ext'  : os.path.join(CSV_ROOT, '2025 Dataset 1 - Same 22 MP Types - Add 40 Spectra'),
    '2025_new'  : os.path.join(CSV_ROOT, '2025 Dataset 2 - New 9 MP Types - 50 Clean and 100 Noisy'),
}
EXCLUDE_FILES = {'ref.csv', 'reference.csv', 'background.csv', 'bg.csv'}

def is_noisy_csv(filepath):
    p = str(filepath).lower()
    if any(k in p for k in ['noisy', '_sd', '-sd', 'sd_']): return True
    if any(k in p for k in ['clean', '_rm', '-rm', 'rm_']): return False
    return False

def extract_label_csv(filepath):
    name = Path(filepath).stem.upper()
    for pattern in ['_SD_', '_RM_', '_NOISY', '_CLEAN', 'PARTICLE', '-NOISY',
                    '-CLEAN', '_50', '_60', '_40', '_100', '_10', '_30',
                    ' SPECTRUMS', ' SPECTUMS', 'ADD_40', '-ADD_40']:
        name = name.replace(pattern, ' ')
    name = re.sub(r'\d+', '', name)
    name = re.sub(r'\bNEW\b|\bJAN\b|\bX\b', '', name)
    name = ' '.join(name.replace('_', ' ').replace('-', ' ').split())
    MAPPING = {
        'NYLON PARTICLE' : 'NYLON', 'PTEE' : 'PTFE', 'PTFE' : 'PTFE',
        'PF THERMOPLASTIC CLEAN' : 'PF THERMOPLASTIC',
        'PF THERMOSET CLEAN'     : 'PF THERMOSET',
    }
    if name in MAPPING: return MAPPING[name]
    if name in ASSUMED_CLASSES: return name
    for cls in ASSUMED_CLASSES:
        if cls in name or name in cls: return cls
    return None

def read_csv_multispectra(filepath, sep=','):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        start_idx = 0
        for i, line in enumerate(lines):
            parts = line.strip().split(sep)
            if len(parts) >= 2:
                try:
                    float(parts[0].replace(',', '.'))
                    start_idx = i; break
                except ValueError: continue
        valid = ''.join(lines[start_idx:])
        headers = lines[start_idx-1].strip().split(sep) if start_idx > 0 else []
        try:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal=',')
        except Exception:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal='.')
        cols = []
        for ci, cn in enumerate(df.columns):
            h = headers[ci].upper() if ci < len(headers) else ''
            v = str(df[cn].iloc[0]).upper()
            if any(k in h for k in ['AIR','BACKGROUND','BG']): continue
            if any(k in v for k in ['AIR','BACKGROUND','BG']): continue
            cols.append(cn)
        df = df[cols].apply(pd.to_numeric, errors='coerce')
        df = df.dropna(subset=[df.columns[0]])
        if len(df) < 100: return None
        wn    = df.iloc[:, 0].values.astype(float)
        order = np.argsort(wn); wn = wn[order]
        spectra = []
        for c in range(1, df.shape[1]):
            ab = df.iloc[order, c].values.astype(float)
            if np.isnan(ab).all() or ab.std() < 1e-10: continue
            nans = np.isnan(ab)
            if nans.any():
                ab[nans] = np.interp(np.where(nans)[0], np.where(~nans)[0], ab[~nans])
            spectra.append(ab.astype(np.float32))
        return (wn, spectra) if spectra else None
    except Exception: return None

print('✓ Fonctions de lecture définies')

✓ Fonctions de lecture définies


In [9]:
print('Chargement des CSV bruités (pour TEST uniquement)...')
csv_records = []
for src_name, folder in PATHS_CSV.items():
    if not os.path.exists(folder): continue
    files = glob.glob(os.path.join(folder, '**/*.csv'), recursive=True)
    for fp in files:
        if Path(fp).name.lower() in EXCLUDE_FILES: continue
        label = extract_label_csv(fp)
        if label is None: continue
        result = read_csv_multispectra(fp)
        if result is None: continue
        wn, spectra_list = result
        if not is_noisy_csv(fp): continue
        for sp in spectra_list:
            sp_interp = np.interp(WN_GRID, wn, sp).astype(np.float32)
            csv_records.append({'label': label, 'spectrum': sp_interp})

df_csv_noisy = pd.DataFrame(csv_records)
df_csv_noisy['label_enc'] = le.transform(df_csv_noisy['label'])

# ── Même split fixe que tous les autres notebooks — on ne garde QUE le Test ──
csv_noisy_counts = Counter(df_csv_noisy['label_enc'])
csv_singleton = {k for k, v in csv_noisy_counts.items() if v < 3}
df_csv_multi  = df_csv_noisy[~df_csv_noisy['label_enc'].isin(csv_singleton)]

idx_tr_csv, idx_valtest_csv = train_test_split(
    range(len(df_csv_multi)), test_size=0.3, random_state=SEED, stratify=df_csv_multi['label_enc'])
idx_val_csv, idx_test_csv = train_test_split(idx_valtest_csv, test_size=0.5, random_state=SEED)

df_csv_test = df_csv_multi.iloc[idx_test_csv].reset_index(drop=True)

print(f'✓ CSV Test (FIXE, identique aux autres notebooks) : {len(df_csv_test)} spectres')
print('  (Le train/val CSV ne sont PAS chargés — inutiles pour ce test)')

Chargement des CSV bruités (pour TEST uniquement)...
✓ CSV Test (FIXE, identique aux autres notebooks) : 400 spectres
  (Le train/val CSV ne sont PAS chargés — inutiles pour ce test)


---
## 🏗️ Section 3 — Datasets (Train/Val = .npy SEULEMENT ; Test = .npy ET CSV)


In [10]:
class SimpleMultiTaskDataset(Dataset):
    def __init__(self, noisy, clean, labels):
        self.noisy  = noisy.astype(np.float32)
        self.clean  = clean.astype(np.float32)
        self.labels = labels.astype(np.int64)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        x       = torch.tensor(self.noisy[idx], dtype=torch.float32)
        x_clean = torch.tensor(self.clean[idx], dtype=torch.float32)
        y       = torch.tensor(self.labels[idx], dtype=torch.long)
        mu, sigma = x.mean(), x.std() + 1e-8
        x       = (x - mu) / sigma
        x_clean = (x_clean - mu) / sigma
        return x, x_clean, y

# ── Train/Val : .npy SEULEMENT ────────────────────────────────────────────
train_dataset = SimpleMultiTaskDataset(X_train_noisy, X_train_clean, y_train)
val_dataset   = SimpleMultiTaskDataset(X_val_noisy,   X_val_clean,   y_val)

# ── Test : .npy officiel ET CSV, séparément ────────────────────────────────
test_npy_dataset = SimpleMultiTaskDataset(npy_test_noisy, npy_test_clean, labels_test)

csv_test_spectra = np.stack(df_csv_test['spectrum'].values)
# Cible de denoising CSV = le spectre lui-même (on ne calcule PAS de proxy,
# ce test n'entraîne jamais sur du CSV donc pas besoin d'une vraie cible propre —
# on l'utilise seulement pour la classification en évaluation)
test_csv_dataset = SimpleMultiTaskDataset(csv_test_spectra, csv_test_spectra,
                                          df_csv_test['label_enc'].values)

train_loader    = DataLoader(train_dataset, batch_size=CFG['batch_size'], shuffle=True, num_workers=0)
val_loader      = DataLoader(val_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
test_npy_loader = DataLoader(test_npy_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
test_csv_loader = DataLoader(test_csv_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)

print(f'✓ train_dataset (.npy SEUL) : {len(train_dataset)}')
print(f'✓ val_dataset   (.npy SEUL) : {len(val_dataset)}')
print(f'✓ test_npy_dataset : {len(test_npy_dataset)}')
print(f'✓ test_csv_dataset : {len(test_csv_dataset)}  (jamais vu pendant Phase 2/3)')

✓ train_dataset (.npy SEUL) : 17100
✓ val_dataset   (.npy SEUL) : 900
✓ test_npy_dataset : 18000
✓ test_csv_dataset : 400  (jamais vu pendant Phase 2/3)


---
## 🏛️ Section 4 — Architecture (identique au backbone SSL mixte)


In [11]:
class PatchEmbedding(nn.Module):
    def __init__(self, L, patch_size, stride, d_model):
        super().__init__()
        self.P, self.S, self.D = patch_size, stride, d_model
        self.N = (L - patch_size) // stride + 2
        self.patch_proj = nn.Linear(patch_size, d_model)
        self.pos_embed  = nn.Embedding(self.N, d_model)
        self.dropout    = nn.Dropout(0.1)
    def get_raw_patches(self, x):
        B = x.shape[0]
        pad = x[:, -1:].expand(B, self.S)
        x_pad = torch.cat([x, pad], dim=1)
        return x_pad.unfold(1, self.P, self.S)
    def forward(self, x):
        patches  = self.get_raw_patches(x)
        content  = self.patch_proj(patches)
        pos_vecs = self.pos_embed(torch.arange(self.N, device=x.device))
        return self.dropout(content + pos_vecs)

class ConformerFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.W1, self.V, self.W2 = (nn.Linear(d_model, d_ff), nn.Linear(d_model, d_ff),
                                     nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.norm(x)
        return self.drop(self.W2(F.silu(self.W1(x)) * self.V(x)))

class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.pw1  = nn.Conv1d(d_model, 2*d_model, 1)
        self.glu  = nn.GLU(dim=1)
        self.dw   = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size//2, groups=d_model)
        self.bn   = nn.BatchNorm1d(d_model)
        self.act  = nn.SiLU()
        self.pw2  = nn.Conv1d(d_model, d_model, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        r = x
        x = self.norm(x).transpose(1,2)
        x = self.glu(self.pw1(x))
        x = self.act(self.bn(self.dw(x)))
        x = self.drop(self.pw2(x)).transpose(1,2)
        return r + x

class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, kernel_size=31):
        super().__init__()
        self.ffn1 = ConformerFFN(d_model, d_ff, dropout)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.conv = ConformerConvModule(d_model, kernel_size, dropout)
        self.ffn2 = ConformerFFN(d_model, d_ff, dropout)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)
        xn = self.attn_norm(x)
        x  = x + self.drop(self.attn(xn, xn, xn)[0])
        x  = self.conv(x)
        x  = x + 0.5 * self.ffn2(x)
        return self.norm(x)

class TransformerBackbone(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        for l in self.layers: x = l(x)
        return self.norm(x)

class ClassificationHead(nn.Module):
    def __init__(self, d_model, n_classes, dropout=0.1, hidden_dim=None):
        super().__init__()
        if hidden_dim is None: hidden_dim = d_model // 2
        self.attn_pool = nn.Linear(d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, n_classes))
    def forward(self, z):
        w = F.softmax(self.attn_pool(z), dim=1)
        return self.head((w * z).sum(dim=1))

class DenoisingHead(nn.Module):
    def __init__(self, d_model, patch_size, n_patches, stride, spectrum_length):
        super().__init__()
        self.P, self.S, self.N, self.L = patch_size, stride, n_patches, spectrum_length
        self.proj = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                   nn.GELU(), nn.Linear(d_model, patch_size))
    def forward(self, z):
        B = z.shape[0]
        pr = self.proj(z)
        out = torch.zeros(B, self.L+self.S, device=z.device)
        cnt = torch.zeros(self.L+self.S, device=z.device)
        for k in range(self.N):
            s = k * self.S
            out[:, s:s+self.P] += pr[:, k, :]
            cnt[s:s+self.P]    += 1
        return (out / cnt.clamp(min=1))[:, :self.L]

class PatchTSTMultiTask(nn.Module):
    def __init__(self, patch_embed, backbone, class_head, denoise_head, alpha=1.0, beta=0.5):
        super().__init__()
        self.patch_embed, self.backbone = patch_embed, backbone
        self.class_head, self.denoise_head = class_head, denoise_head
        self.alpha, self.beta = alpha, beta
    def encode(self, x):
        return self.backbone(self.patch_embed(x))
    def classify(self, x):
        return self.class_head(self.encode(x))
    def forward(self, x, y=None, clean_target=None):
        z = self.encode(x)
        logits   = self.class_head(z)
        denoised = self.denoise_head(z)
        loss = None
        if y is not None and clean_target is not None:
            loss_clf = F.cross_entropy(logits, y, label_smoothing=0.1)
            loss_den = F.mse_loss(denoised, clean_target)
            loss = self.alpha * loss_clf + self.beta * loss_den
        return logits, denoised, loss

print('✓ Architecture définie')

✓ Architecture définie


---
## 📦 Section 5 — Charger le backbone SSL MIXTE (déjà entraîné, pas de nouveau Phase 1)


In [12]:
patch_embed  = PatchEmbedding(CFG['L'], CFG['patch_size'], CFG['stride'], CFG['d_model']).to(DEVICE)
backbone     = TransformerBackbone(CFG['d_model'], CFG['n_heads'], CFG['n_layers'],
                                    CFG['d_ff'], CFG['dropout']).to(DEVICE)

ckpt = torch.load(CFG['ssl_path'], map_location=DEVICE, weights_only=False)
patch_embed.load_state_dict(ckpt['patch_embed'])
backbone.load_state_dict(ckpt['backbone'])
print(f'✓ Backbone SSL MIXTE chargé (entraîné sur .npy+CSV en Phase 1)')
print(f'  Val MSE au moment de la sauvegarde : {ckpt["ssl_val_loss"]:.6f}')

class_head   = ClassificationHead(CFG['d_model'], CFG['N_CLASSES'], dropout=0.1).to(DEVICE)
denoise_head = DenoisingHead(CFG['d_model'], CFG['patch_size'], N_PATCHES,
                              CFG['stride'], CFG['L']).to(DEVICE)

total = sum(p.numel() for m in [patch_embed, backbone, class_head, denoise_head]
            for p in m.parameters() if p.requires_grad)
print(f'Total paramètres : {total:,}')

✓ Backbone SSL MIXTE chargé (entraîné sur .npy+CSV en Phase 1)
  Val MSE au moment de la sauvegarde : 0.002125
Total paramètres : 6,596,191


---
## 🛠️ Section 6 — Fonctions d'entraînement


In [13]:
def snr_db(clean, signal):
    noise_power  = ((signal - clean) ** 2).mean(dim=-1) + 1e-8
    signal_power = (clean ** 2).mean(dim=-1) + 1e-8
    return 10 * torch.log10(signal_power / noise_power)

def multitask_train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    total_mse, total_snr = 0.0, 0.0
    for x, x_clean, y in loader:
        x, x_clean, y = x.to(DEVICE), x_clean.to(DEVICE), y.to(DEVICE)
        logits, denoised, loss = model(x, y=y, clean_target=x_clean)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        with torch.no_grad():
            mse = F.mse_loss(denoised, x_clean)
            snr_gain = (snr_db(x_clean, denoised) - snr_db(x_clean, x)).mean()
        total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item()
        total_mse += mse.item()*len(y); total_snr += snr_gain.item()*len(y); n += len(y)
    return total_loss/n, correct/n, total_mse/n, total_snr/n

@torch.no_grad()
def multitask_eval_epoch(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    total_mse, total_snr = 0.0, 0.0
    for x, x_clean, y in loader:
        x, x_clean, y = x.to(DEVICE), x_clean.to(DEVICE), y.to(DEVICE)
        logits, denoised, loss = model(x, y=y, clean_target=x_clean)
        mse = F.mse_loss(denoised, x_clean)
        snr_gain = (snr_db(x_clean, denoised) - snr_db(x_clean, x)).mean()
        total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item()
        total_mse += mse.item()*len(y); total_snr += snr_gain.item()*len(y); n += len(y)
    return total_loss/n, correct/n, total_mse/n, total_snr/n

@torch.no_grad()
def eval_accuracy_only(model, loader):
    """Pour CSV où on n'a pas de vraie cible de denoising fiable."""
    model.eval()
    correct, n = 0, 0
    for x, x_clean, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.classify(x)
        correct += (logits.argmax(1)==y).sum().item(); n += len(y)
    return correct/n

print('✓ Fonctions définies')

✓ Fonctions définies


---
## 🔍 Phase 2 — Linear Probing sur .npy SEULEMENT


In [14]:
for p in patch_embed.parameters(): p.requires_grad = False
for p in backbone.parameters():    p.requires_grad = False

probe_model = PatchTSTMultiTask(patch_embed, backbone, class_head, denoise_head,
                                 alpha=CFG['alpha'], beta=CFG['beta']).to(DEVICE)
probe_optimizer = AdamW(filter(lambda p: p.requires_grad, probe_model.parameters()),
                        lr=CFG['probe_lr'], weight_decay=1e-4)

best_probe_acc = 0.0
print(f'=== Phase 2 : Linear Probing SUR .npy SEUL ({CFG["probe_epochs"]} époques) ===')
print(f'{"Époque":>7} | {"Tr.Acc":>7} | {"Val.Acc":>8}')
print('-'*35)

for epoch in range(1, CFG['probe_epochs']+1):
    tr_loss, tr_acc, tr_mse, tr_snr = multitask_train_epoch(probe_model, train_loader, probe_optimizer)
    va_loss, va_acc, va_mse, va_snr = multitask_eval_epoch(probe_model, val_loader)

    safe_log(writer, 'Phase2_Probe/Accuracy', {'Train': tr_acc, 'Val': va_acc}, epoch, method='add_scalars')

    if va_acc > best_probe_acc:
        best_probe_acc = va_acc
        torch.save(probe_model.state_dict(), CFG['probe_path'])

    if epoch % 20 == 0 or epoch == 1:
        print(f'{epoch:7d} | {tr_acc:6.2%} | {va_acc:7.2%}')

print(f'\n✓ Meilleure Val Acc Phase 2 : {best_probe_acc:.2%}')

=== Phase 2 : Linear Probing SUR .npy SEUL (80 époques) ===
 Époque |  Tr.Acc |  Val.Acc
-----------------------------------
      1 | 12.81% |  35.33%
     20 | 60.26% |  84.67%
     40 | 65.18% |  86.78%
     60 | 68.09% |  86.89%
     80 | 70.46% |  88.89%

✓ Meilleure Val Acc Phase 2 : 90.11%


---
## 🎯 Phase 3 — Fine-tuning sur .npy SEULEMENT


In [15]:
probe_model.load_state_dict(torch.load(CFG['probe_path'], map_location=DEVICE, weights_only=False))
for p in probe_model.parameters(): p.requires_grad = True

ft_optimizer = AdamW([
    {'params': probe_model.patch_embed.parameters(),  'lr': CFG['ft_lr']},
    {'params': probe_model.backbone.parameters(),     'lr': CFG['ft_lr']},
    {'params': probe_model.class_head.parameters(),   'lr': CFG['ft_lr']*10},
    {'params': probe_model.denoise_head.parameters(), 'lr': CFG['ft_lr']*10},
], weight_decay=1e-4)
ft_scheduler = CosineAnnealingLR(ft_optimizer, T_max=CFG['ft_epochs'], eta_min=1e-6)

best_ft_acc = 0.0
print(f'=== Phase 3 : Fine-tuning SUR .npy SEUL ({CFG["ft_epochs"]} époques) ===')
print(f'{"Époque":>7} | {"Tr.Acc":>7} | {"Val.Acc":>8}')
print('-'*35)

for epoch in range(1, CFG['ft_epochs']+1):
    tr_loss, tr_acc, tr_mse, tr_snr = multitask_train_epoch(probe_model, train_loader, ft_optimizer)
    va_loss, va_acc, va_mse, va_snr = multitask_eval_epoch(probe_model, val_loader)
    ft_scheduler.step()

    safe_log(writer, 'Phase3_FT/Accuracy', {'Train': tr_acc, 'Val': va_acc}, epoch, method='add_scalars')

    if epoch % 5 == 0:
        torch.save({'model_state': probe_model.state_dict(), 'epoch': epoch, 'cfg': CFG},
                   os.path.join(HOME, 'models', 'checkpoint_sslmixed_npyonly_latest.pth'))

    if va_acc > best_ft_acc:
        best_ft_acc = va_acc
        torch.save({'model_state': probe_model.state_dict(), 'cfg': CFG,
                    'le_classes': le.classes_, 'val_acc': best_ft_acc}, CFG['final_path'])

    if epoch % 10 == 0 or epoch == 1:
        print(f'{epoch:7d} | {tr_acc:6.2%} | {va_acc:7.2%}')

writer.flush()
print(f'\n✓ Meilleure Val Acc Phase 3 : {best_ft_acc:.2%}')
print(f'✓ Sauvegardé → {CFG["final_path"]}')
print('🔴 TÉLÉCHARGE ce fichier vers ton Mac')

=== Phase 3 : Fine-tuning SUR .npy SEUL (80 époques) ===
 Époque |  Tr.Acc |  Val.Acc
-----------------------------------
      1 | 72.62% |  90.89%
     10 | 85.08% |  93.44%
     20 | 88.87% |  94.00%
     30 | 90.91% |  95.56%
     40 | 91.80% |  96.00%
     50 | 92.96% |  96.00%
     60 | 93.29% |  96.33%
     70 | 93.38% |  96.33%
     80 | 93.58% |  96.11%

✓ Meilleure Val Acc Phase 3 : 96.56%
✓ Sauvegardé → /home/glider/models/final_model_sslmixed_npyonly.pth
🔴 TÉLÉCHARGE ce fichier vers ton Mac


---
## 📊 Évaluation finale — LE TEST CLÉ : ce modèle généralise-t-il sur CSV ?


In [16]:
ckpt = torch.load(CFG['final_path'], map_location=DEVICE, weights_only=False)
probe_model.load_state_dict(ckpt['model_state'])
probe_model.eval()

# ── Test .npy officiel (avec denoising, cible fiable) ──────────────────────
all_preds, all_labels = [], []
all_snr_before, all_snr_after = [], []
with torch.no_grad():
    for x, x_clean, y in test_npy_loader:
        x, x_clean = x.to(DEVICE), x_clean.to(DEVICE)
        logits, denoised, _ = probe_model(x)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y.numpy())
        all_snr_before.append(snr_db(x_clean, x).cpu().numpy())
        all_snr_after.append(snr_db(x_clean, denoised).cpu().numpy())

acc_npy = accuracy_score(all_labels, all_preds)
snr_gain_npy = (np.concatenate(all_snr_after) - np.concatenate(all_snr_before)).mean()

# ── Test CSV (classification uniquement — jamais entraîné dessus) ─────────
acc_csv = eval_accuracy_only(probe_model, test_csv_loader)

print('═'*70)
print('  RÉSULTAT — SSL Mixte, Phase 2/3 sur .npy SEUL')
print('═'*70)
print(f'  Test .npy (officiel) : {acc_npy:.2%}   SNR : {snr_gain_npy:+.2f}dB')
print(f'  Test CSV (jamais vu, ni en SSL classif ni en Phase 2/3) : {acc_csv:.2%}')
print('═'*70)
print()
print('  Comparaison à trois termes :')
print(f'    Modèle .npy pur   (SSL npy,     Phase2/3 npy)     : Test npy=95.25%  Test CSV=23.47%')
print(f'    Modèle Mixte      (SSL npy+CSV, Phase2/3 npy+CSV) : Test npy=94.62%  Test CSV=99.25%')
print(f'    CE MODÈLE         (SSL npy+CSV, Phase2/3 npy SEUL): Test npy={acc_npy:.2%}  Test CSV={acc_csv:.2%}')

══════════════════════════════════════════════════════════════════════
  RÉSULTAT — SSL Mixte, Phase 2/3 sur .npy SEUL
══════════════════════════════════════════════════════════════════════
  Test .npy (officiel) : 95.08%   SNR : +10.87dB
  Test CSV (jamais vu, ni en SSL classif ni en Phase 2/3) : 26.75%
══════════════════════════════════════════════════════════════════════

  Comparaison à trois termes :
    Modèle .npy pur   (SSL npy,     Phase2/3 npy)     : Test npy=95.25%  Test CSV=23.47%
    Modèle Mixte      (SSL npy+CSV, Phase2/3 npy+CSV) : Test npy=94.62%  Test CSV=99.25%
    CE MODÈLE         (SSL npy+CSV, Phase2/3 npy SEUL): Test npy=95.08%  Test CSV=26.75%


In [17]:
# ── Conclusion automatique ──────────────────────────────────────────────
if acc_csv > 0.70:
    verdict = "✓ L'exposition SSL seule (sans jamais classifier de CSV) SUFFIT largement\n" \
              "  à éviter l'effondrement du domain gap — résultat très intéressant."
elif acc_csv > 0.40:
    verdict = "~ L'exposition SSL aide PARTIELLEMENT, mais ne suffit pas complètement\n" \
              "  à égaler le modèle mixte complet."
else:
    verdict = "⚠️  L'exposition SSL seule NE SUFFIT PAS — la supervision explicite\n" \
              "  sur CSV (Phase 2/3) reste nécessaire pour éviter le domain gap."

print('═'*65)
print('  CONCLUSION')
print('═'*65)
print(verdict)
print('═'*65)

═════════════════════════════════════════════════════════════════
  CONCLUSION
═════════════════════════════════════════════════════════════════
⚠️  L'exposition SSL seule NE SUFFIT PAS — la supervision explicite
  sur CSV (Phase 2/3) reste nécessaire pour éviter le domain gap.
═════════════════════════════════════════════════════════════════


In [18]:
from torch.utils.tensorboard import SummaryWriter as SW2
HPARAMS_SUGGESTION1 = os.path.join(HOME, 'runs', 'hparams_sarun_suggestion1')

def log_run(cfg, metrics, run_label):
    run_dir = os.path.join(HPARAMS_SUGGESTION1, run_label)
    w = SW2(run_dir)
    hparams_clean = {k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool))}
    w.add_hparams(hparams_clean, metrics)
    w.close()

log_run(
    cfg={'ssl_domain': 'mixed_npy_csv', 'phase23_domain': 'npy_only'},
    metrics={'test_npy_acc': acc_npy, 'test_csv_acc': acc_csv},
    run_label='ssl_mixed_phase23_npy_only',
)
print('✓ Run loggé')
print(f'Dashboard : tensorboard --logdir {HPARAMS_SUGGESTION1} --port 6013')

✓ Run loggé
Dashboard : tensorboard --logdir /home/glider/runs/hparams_sarun_suggestion1 --port 6013
